# Tutorial 11: Graph Neural Network Forward Model

In this tutorial we build a **Graph Neural Network (GNN)** that predicts Hamiltonian
parameters from a quantum-circuit layout.  The key ideas:

1. Every component (qubit, resonator) becomes a **node** in a circuit graph.
2. Each node carries a **static feature vector** built from:
   - **Layer stack** `(thickness, permittivity)` ordered bottom-to-top
   - **Design parameters** — the FULL set from the component JSON defaults, with swept values overridden
   - **Area / perimeter** — from geometric equations in enriched JSONs
   - **Port vector** — `[connector, mwave, o2g, RLC, LumpedPort]`
3. Port connectivity defines **edges** (e.g. qubit ↔ cavity_claw).
4. A GCN performs message passing → graph-level attention pooling → readout MLP → ŷ.

The pipeline: **Dataset → Component Split → Netlist → Static Features → GNN Training → Evaluation**

## 1. Imports

In [1]:
import numpy as np
import pandas as pd

from squadds.ml.graph import (
    build_vocab,
    CircuitGraphBuilder,
    SQuADDSGraphDataset,
    GraphForwardModel,
    GraphTrainer,
    plot_predictions,
)

import warnings
warnings.filterwarnings('ignore')

 /Users/shanto/LFL/SQuADDS_Refactor/.venv/lib/python3.11/site-packages/tqdm/auto.py: 21INFO:datasets:TensorFlow version 2.21.0 available.


## 2. Load the training data

We reuse the same parquet dataset from Tutorial 8.  Each row contains the
swept design parameters and the corresponding Hamiltonian targets.

In [2]:
df = pd.read_parquet("data/training_data.parquet")
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head(3)

Rows: 133,176
Columns: ['claw_length', 'cavity_frequency_GHz', 'kappa_kHz', 'EC', 'EJ', 'qubit_frequency_GHz', 'anharmonicity_MHz', 'g_MHz', 'cross_length', 'cross_gap', 'ground_spacing', 'coupling_length', 'total_length']


,claw_length,cavity_frequency_GHz,kappa_kHz,EC,EJ,qubit_frequency_GHz,anharmonicity_MHz,g_MHz,cross_length,cross_gap,ground_spacing,coupling_length,total_length
0,160.0,8.963333,282.985474,0.119465,16.346243,3.829124,-128.92902,52.250558,310.0,30.0,10.0,200.0,2700.0
1,160.0,6.911806,689.394209,0.119465,16.346243,3.829124,-128.92902,40.291451,310.0,30.0,10.0,500.0,3400.0
2,160.0,8.968642,205.609615,0.119465,16.346243,3.829124,-128.92902,52.281505,310.0,30.0,10.0,200.0,2700.0


## 3. Define targets and column groups

In [3]:
# Hamiltonian targets (what the model predicts)
TARGET_COLS = ['qubit_frequency_GHz', 'anharmonicity_MHz',
               'cavity_frequency_GHz', 'kappa_kHz', 'g_MHz']
TARGET_NAMES = ['f_q', 'alpha', 'f_r', 'kappa', 'g']

# Design parameters swept in the simulations
QUBIT_SWEPT   = ['cross_length', 'cross_gap', 'ground_spacing']  # TransmonCross
CAVITY_SWEPT  = ['claw_length', 'coupling_length', 'total_length']  # CavityClaw

print(f"Targets:  {TARGET_COLS}")
print(f"Qubit swept:  {QUBIT_SWEPT}")
print(f"Cavity swept: {CAVITY_SWEPT}")

Targets:  ['qubit_frequency_GHz', 'anharmonicity_MHz', 'cavity_frequency_GHz', 'kappa_kHz', 'g_MHz']
Qubit swept:  ['cross_length', 'cross_gap', 'ground_spacing']
Cavity swept: ['claw_length', 'coupling_length', 'total_length']


## 4. Build the parameter-key vocabulary

The vocabulary maps every unique design-parameter name across all enriched
component JSONs to an integer ID.  Position 0 is reserved for `<PAD>`.

In [4]:
import os

# Path to the CavityClaw JSON we generated
EXTRA_JSON_DIR = os.path.join(
    os.path.dirname(os.path.abspath('.')),
    'squadds', 'ml', 'graph', 'component_data',
)

# Try the repo root if the relative path doesn't work
if not os.path.isdir(EXTRA_JSON_DIR):
    EXTRA_JSON_DIR = os.path.join(
        os.path.expanduser('~/LFL/SQuADDS_Refactor'),
        'squadds', 'ml', 'graph', 'component_data',
    )

vocab = build_vocab(
    extra_jsons=[os.path.join(EXTRA_JSON_DIR, 'CavityClaw.json')],
)
print(f"Vocabulary size: {len(vocab)} parameter keys")
print(f"Sample keys: {list(vocab.keys())[:10]}")

Vocabulary size: 258 parameter keys
Sample keys: ['<PAD>', 'JJ_gap', 'JJ_height', 'JJ_length', 'JJ_pad_lower_height', 'JJ_pad_lower_pos_x', 'JJ_pad_lower_pos_y', 'JJ_pad_lower_width', 'JJ_width', '_default_connection_pads.claw_cpw_length']


## 5. Define the circuit netlist

For each row we define a two-node circuit graph:

| Node | Component | Port vector | Role |
|------|-----------|-------------|------|
| 0 | `TransmonCross` | `[1, 0, 2, 0, 1]` | Qubit |
| 1 | `CavityClaw` | `[1, 2, 0, 0, 0]` | Cavity + coupler |

Port vector layout: `[connector, mwave, o2g, RLC, LumpedPort]`

Connectivity:  **qubit.south ↔ cavity_claw.north** → edge (0, 1)

In [5]:
# Shared layer stack for all data points (Si substrate + Al metal)
LAYER_STACK = [(350.0, 11.45), (0.25, 0.0)]

# Port vectors
QUBIT_PORTS  = [1, 0, 2, 0, 1]   # 1 connector (claw), 0 mwave, 2 o2g, 0 RLC, 1 LumpedPort
CAVITY_PORTS = [1, 2, 0, 0, 0]   # 1 connector, 2 mwave (feedline ports), 0 o2g, 0 RLC, 0 LumpedPort

# Connectivity: qubit ↔ cavity
EDGES = [(0, 1)]

print("Netlist defined:")
print(f"  Node 0: TransmonCross  ports={QUBIT_PORTS}")
print(f"  Node 1: CavityClaw     ports={CAVITY_PORTS}")
print(f"  Edges:  {EDGES}")
print(f"  Layer stack: {LAYER_STACK}")

Netlist defined:
  Node 0: TransmonCross  ports=[1, 0, 2, 0, 1]
  Node 1: CavityClaw     ports=[1, 2, 0, 0, 0]
  Edges:  [(0, 1)]
  Layer stack: [(350.0, 11.45), (0.25, 0.0)]


## 6. Build circuit graphs from the dataset

For each row, we split the swept columns into per-component
`design_overrides`.  The `CircuitGraphBuilder` loads **all** default params
from the component JSONs and overrides the swept values.

In [6]:
builder = CircuitGraphBuilder(
    vocab=vocab,
    k_max=20,
    n_ls=5,
    extra_json_dir=EXTRA_JSON_DIR,
)

graphs = []
for _, row in df.iterrows():
    # Split swept params into per-component override dicts
    qubit_overrides = {k: f"{row[k]}um" for k in QUBIT_SWEPT}
    cavity_overrides = {k: f"{row[k]}um" for k in CAVITY_SWEPT}

    # Build the graph
    g = builder.build(
        components=[
            {
                'type': 'TransmonCross',
                'design_overrides': qubit_overrides,
                'layer_stack': LAYER_STACK,
                'ports_vector': QUBIT_PORTS,
            },
            {
                'type': 'CavityClaw',
                'design_overrides': cavity_overrides,
                'layer_stack': LAYER_STACK,
                'ports_vector': CAVITY_PORTS,
            },
        ],
        edges=EDGES,
        targets=[row[c] for c in TARGET_COLS],
    )
    graphs.append(g)

print(f"Built {len(graphs):,} circuit graphs")
print(f"Feature dim per node: {graphs[0].x.shape[1]}")
print(f"Nodes per graph: {graphs[0].x.shape[0]}")

Built 133,176 circuit graphs
Feature dim per node: 57
Nodes per graph: 2


## 7. Train / Val / Test split

In [11]:
dataset = SQuADDSGraphDataset(graphs, val_split=0.1, test_split=0.1, seed=42)
print(f"Train : {len(dataset.train_graphs):,}")
print(f"Val   : {len(dataset.val_graphs):,}")
print(f"Test  : {len(dataset.test_graphs):,}")

Train : 106,542
Val   : 13,317
Test  : 13,317


## 8. Build and inspect the GNN model

Architecture:
```
Raw node features → UnpackNodeFeatures → NodeEncoder → E_static (N, 128)
                                                            │
                                                     GCNConvK3 × 2 ← adjacency
                                                            │
                                                   GlobalAttentionPoolK3
                                                            │
                                                     Readout MLP → ŷ (5,)
```

In [7]:
model_builder = GraphForwardModel(
    vocab_size=len(vocab),
    embed_dim=32,
    node_latent_dim=128,
    n_gcn_layers=2,
    n_targets=len(TARGET_COLS),
    k_max=20,
    n_ls=5,
    readout_dim=64,
    dropout_rate=0.1,
    aggregation='deepsets',  # or 'sum'
)

model = model_builder.build()
model.summary()

Model: "graph_forward_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x_in (InputLayer)   │ (None, 57)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ unpack              │ [(None, 5, 2),    │          0 │ x_in[0][0]        │
│ (UnpackNodeFeature… │ (None, 20),       │            │                   │
│                     │ (None, 20),       │            │                   │
│                     │ (None, 1), (None, │            │                   │
│                     │ 1), (None, 5)]    │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ node_encoder        │ (None, 128)       │     59,456 │ unpack[0][0],     │
│ (NodeEncoder)       │                   │            │ unpack[0][1],     │
│                     │                   │            │ unpack[0][2],     │
│                     │                   │            │ unpack[0][3],     │
│                     │                   │            │ unpack[0][4],     │
│                     │                   │            │ unpack[0][5]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ a_in (InputLayer)   │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gcn_0 (GCNConvK3)   │ (None, 128)       │     16,512 │ node_encoder[0][… │
│                     │                   │            │ a_in[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ gcn_0[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128)       │          0 │ dropout[0][0],    │
│                     │                   │            │ node_encoder[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gcn_1 (GCNConvK3)   │ (None, 128)       │     16,512 │ add[0][0],        │
│                     │                   │            │ a_in[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ gcn_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 128)       │          0 │ dropout_1[0][0],  │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ i_in (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_pool         │ (None, 128)       │     16,641 │ add_1[0][0],      │
│ (GlobalAttentionPo… │                   │            │ i_in[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ readout_1 (Dense)   │ (None, 64)        │      8,256 │ global_pool[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ readout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ readout_out (Dense) │ (None, 5)         │        325 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 117,702 (459.77 KB)

 Trainable params: 117,702 (459.77 KB)

 Non-trainable params: 0 (0.00 B)

## 9. Train the model

In [ ]:
trainer = GraphTrainer(
    model_builder=model_builder,
    learning_rate=5e-4,
    target_names=TARGET_NAMES,
)

history = trainer.train(
    train_graphs=dataset.train_graphs,
    val_graphs=dataset.val_graphs,
    epochs=500,
    batch_size=32,
    patience=50,
    verbose=1,
)

Epoch 1/500
3326/3330 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 32931.2180 - mae: 69.9963
Epoch 1: val_loss improved from None to 665.32867, saving model to saved_models/run_20260309_121556/model.keras

Epoch 1: finished saving model to saved_models/run_20260309_121556/model.keras
3330/3330 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - loss: 7311.7515 - mae: 35.8243 - val_loss: 665.3287 - val_mae: 13.1204 - learning_rate: 5.0000e-04
Epoch 2/500
3328/3330 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1129.3239 - mae: 19.2897
Epoch 2: val_loss improved from 665.32867 to 455.68503, saving model to saved_models/run_20260309_121556/model.keras

Epoch 2: finished saving model to saved_models/run_20260309_121556/model.keras
3330/3330 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 1071.8530 - mae: 18.4235 - val_loss: 455.6850 - val_mae: 10.6521 - learning_rate: 5.0000e-04
Epoch 3/500
3328/3330 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 948.8623 - mae: 16.3035
Epoch 3: val_loss improved from 455.68503 to 447.20529, s

## 10. Training curves

In [ ]:
# matplotlib inline

In [ ]:
# matplotlib inline

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['loss'], label='Train')
if 'val_loss' in history:
    ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.set_title('Loss')
ax1.legend()
ax1.set_yscale('log')

ax2.plot(history['mae'], label='Train')
if 'val_mae' in history:
    ax2.plot(history['val_mae'], label='Val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.set_title('Mean Absolute Error')
ax2.legend()

fig.tight_layout()
plt.show()

## 11. Evaluate on test set

In [ ]:
metrics = trainer.evaluate(dataset.test_graphs)

print(f"{'Target':<10} {'R²':>8} {'RMSE':>10} {'MAE':>10}")
print('-' * 40)
for name, m in metrics.items():
    print(f"{name:<10} {m['r2']:>8.4f} {m['rmse']:>10.4f} {m['mae']:>10.4f}")

## 12. Parity plots

In [ ]:
y_pred = trainer.predict(dataset.test_graphs)
y_true = np.array([g.y for g in dataset.test_graphs])

fig = plot_predictions(y_true, y_pred, target_names=TARGET_NAMES)
plt.show()

## 13. Transfer learning demo

The graph architecture is **component-agnostic** — the same model can
handle a different qubit geometry by simply changing the component type
and design overrides.  Here we show how to create graphs with a
hypothetical different component, demonstrating that the model forward
pass works without retraining.

> **Note:** The predictions won't be accurate on unseen component types
> without fine-tuning, but the architecture remains valid.

In [ ]:
# Same model, different component overrides
transfer_graph = builder.build(
    components=[
        {
            'type': 'TransmonCross',  # Same type, different params
            'design_overrides': {'cross_length': '400um', 'cross_gap': '40um', 'ground_spacing': '12um'},
            'layer_stack': LAYER_STACK,
            'ports_vector': QUBIT_PORTS,
        },
        {
            'type': 'CavityClaw',
            'design_overrides': {'claw_length': '200um', 'coupling_length': '300um', 'total_length': '5000um'},
            'layer_stack': LAYER_STACK,
            'ports_vector': CAVITY_PORTS,
        },
    ],
    edges=EDGES,
    targets=[0, 0, 0, 0, 0],  # dummy
)

pred = trainer.predict([transfer_graph])
print("Transfer prediction:")
for name, val in zip(TARGET_NAMES, pred[0]):
    print(f"  {name}: {val:.4f}")

## 14. Save the trained model

In [ ]:
trainer.save('/tmp/graph_forward_model')
print("Model saved to /tmp/graph_forward_model/")

import os
for f in os.listdir('/tmp/graph_forward_model'):
    size = os.path.getsize(f'/tmp/graph_forward_model/{f}')
    print(f"  {f:30s} {size:>10,} bytes")

---

## Summary

| Step | What |
|---|---|
| Vocabulary | All parameter keys from component JSONs → integer IDs |
| Netlist | Per-component: type + design_overrides + layer_stack + ports_vector |
| Graph | `CircuitGraphBuilder.build()` → `spektral.data.Graph` with flat node features |
| Model | `UnpackNodeFeatures` → `NodeEncoder` → `GCNConvK3` → `GlobalAttentionPoolK3` → readout |
| Training | `GraphTrainer.train()` with DisjointLoader, early stopping, LR scheduling |
| Evaluation | Per-target R², RMSE, MAE + parity plots |